In [6]:
import json
from pathlib import Path

CATEGORIES_OF_INTEREST = {'cs.RO', 'cs.LG', 'cs.AI', 'q-bio', 'physics.plasm-ph'}
# Note: q-bio has subcategories like q-bio.BM, q-bio.NC, etc. — 
# you'll want to decide whether to match the prefix or specific subcats
ARXIV_PATH = Path.home() / "datasets" / "arxiv" / "arxiv-metadata-oai-snapshot.json"

filtered = []
with open(ARXIV_PATH, 'r') as f:
    for line in f:
        record = json.loads(line)
        cats = set(record.get('categories', '').split())
        if cats & CATEGORIES_OF_INTEREST or any(
            c.startswith('q-bio') for c in cats
        ):
            filtered.append(record)

print(f"Filtered down to {len(filtered)} papers")

Filtered down to 466351 papers


In [7]:
from collections import Counter

# Category counts (papers can be in multiple)
cat_counter = Counter()
for paper in filtered:
    cats = set(paper['categories'].split())
    for c in cats:
        if c in CATEGORIES_OF_INTEREST or c.startswith('q-bio'):
            cat_counter[c] += 1

print(cat_counter.most_common(10))

[('cs.LG', 263561), ('cs.AI', 174962), ('cs.RO', 52325), ('physics.plasm-ph', 19708), ('q-bio.QM', 13006), ('q-bio.PE', 12907), ('q-bio.NC', 11952), ('q-bio.BM', 6691), ('q-bio.MN', 4131), ('q-bio.GN', 3796)]


In [8]:
from collections import Counter
years = Counter(p['update_date'][:4] for p in filtered)
print(sorted(years.items()))

[('2007', 4730), ('2008', 1326), ('2009', 4101), ('2010', 1845), ('2011', 2387), ('2012', 3684), ('2013', 5149), ('2014', 4902), ('2015', 9130), ('2016', 8690), ('2017', 10175), ('2018', 15138), ('2019', 23108), ('2020', 32887), ('2021', 38549), ('2022', 41545), ('2023', 50381), ('2024', 66740), ('2025', 93540), ('2026', 48344)]


In [9]:
import random
random.seed(42)  # reproducibility

CORE_CATS = {'cs.RO', 'physics.plasm-ph'}
QBIO_PREFIX = 'q-bio'
CAPPED_CATS = {'cs.LG', 'cs.AI'}
CAP = 50_000
YEAR_MIN = 2020

# First, restrict to 2020+
recent = [p for p in filtered if int(p['update_date'][:4]) >= YEAR_MIN]

# Split into "always keep" vs "cappable"
always_keep = []
cappable = []

for paper in recent:
    cats = set(paper['categories'].split())
    in_core = bool(cats & CORE_CATS) or any(c.startswith(QBIO_PREFIX) for c in cats)
    if in_core:
        always_keep.append(paper)
    else:
        # Only in cs.LG and/or cs.AI
        cappable.append(paper)

# Cap the cs.LG/cs.AI-only papers
# Sample 50k per category, dedupe by id
lg_only = [p for p in cappable if 'cs.LG' in p['categories'].split()]
ai_only = [p for p in cappable if 'cs.AI' in p['categories'].split()]

lg_sample = random.sample(lg_only, min(CAP, len(lg_only)))
ai_sample = random.sample(ai_only, min(CAP, len(ai_only)))

# Combine and dedupe by arxiv id
seen_ids = set()
final_corpus = []
for paper in always_keep + lg_sample + ai_sample:
    if paper['id'] not in seen_ids:
        final_corpus.append(paper)
        seen_ids.add(paper['id'])

print(f"Final corpus: {len(final_corpus)} papers")

Final corpus: 176931 papers


In [10]:
import json

OUT_PATH = Path("data") / "corpus.jsonl"
OUT_PATH.parent.mkdir(exist_ok=True)

with open(OUT_PATH, 'w') as f:
    for paper in final_corpus:
        f.write(json.dumps(paper) + '\n')

print(f"Wrote {len(final_corpus)} papers to {OUT_PATH}")

Wrote 176931 papers to data\corpus.jsonl


In [ ]:
import json
import pandas as pd

with open('data/corpus.jsonl') as f:
    corpus = [json.loads(line) for line in f]
# display corpus as df
df = pd.DataFrame(corpus)

display(df.head())

,title,abstract,categories
0,An accurate model for genetic hitch-hiking,We suggest a simple deterministic approximat...,q-bio.PE
1,Generation interval contraction and epidemic d...,The generation interval is the time between ...,q-bio.QM math.PR stat.AP
2,Molecular Dynamics Simulation of Vascular Netw...,Endothelial cells are responsible for the fo...,cond-mat.soft q-bio.CB
3,Eutacticity in sea urchin evolution,"An eutactic star, in a n-dimensional space, ...",q-bio.QM
4,Kinetic dissipation and anisotropic heating in...,The kinetic evolution of the Orszag-Tang vor...,physics.plasm-ph physics.space-ph
